# Segment Anything 2 (SAM2) 

This notebook allows to use the SAM2 model for image segmentation. It was used to create the segmentation masks of the Hypericum perforatum flower dataset. It constructed 384 masks overlays and segmentations in `output_masks/`

## Cloning the SAM2 repository and downloading the model weights

The two following cells must be run only if it is a kaggle notebook. If you are running this notebook locally, you can skip these cells and directly import the SAM2 model.

In [ ]:
# # Clone SAM2
# !git clone https://github.com/facebookresearch/segment-anything-2.git
# %cd segment-anything-2

# # Install SAM2 locally
# !pip install -e .

# # Download checkpoints
# !mkdir -p checkpoints
# !bash checkpoints/download_ckpts.sh

Cloning into 'segment-anything-2'...
remote: Enumerating objects: 1086, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 1086 (delta 6), reused 4 (delta 4), pack-reused 1076 (from 3)
Receiving objects: 100% (1086/1086), 128.20 MiB | 18.14 MiB/s, done.
Resolving deltas: 100% (389/389), done.
/kaggle/working/segment-anything-2
Obtaining file:///kaggle/working/segment-anything-2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.8 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp312-cp312-linux_x86_64.whl size=

In [ ]:
# # download the checkpoints for model
# !chmod +x /kaggle/working/segment-anything-2/checkpoints/download_ckpts.sh
# !/kaggle/working/segment-anything-2/checkpoints/download_ckpts.sh
# !mv *.pt /kaggle/working/segment-anything-2/checkpoints/

--2026-03-14 13:38:29--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 108.158.20.120, 108.158.20.43, 108.158.20.111, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|108.158.20.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156008466 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2.1_hiera_tiny.pt.1’

sam2.1_hiera_tiny.p 100%[===================>] 148.78M   492MB/s    in 0.3s    

2026-03-14 13:38:29 (492 MB/s) - ‘sam2.1_hiera_tiny.pt.1’ saved [156008466/156008466]

--2026-03-14 13:38:29--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 108.158.20.120, 108.158.20.43, 108.158.20.21, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|108.158.20.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285

## Importing libraries

1. Kaggle notebooks case:
    
    >You must have the dataset loaded in the input folder of the Kaggle notebook and replace the line `sys.path.append("/kaggle/input/models/clementmartin1/sam2-utils/other/default/3/")` with the path to your own dataset.

    > To load the SAM_utils file, you have to create a model and add it to the input folder of  the notebook. You can find the SAM_utils file in the `scripts/` folder.

2. Local case:
    
    > You don't need to load this path `/kaggle/input/models/clementmartin1/sam2-utils/other/default/3/` since your dataset should be in the `data/` folder. 

    > You can directly import the SAM2 model and the SAM_utils file. You can find the SAM_utils file in the `scripts/` folder.

In [3]:
import sys
sys.path.append("/kaggle/input/models/clementmartin1/sam2-utils/other/default/3/")

from SAM2_utils import load_sam2_model, process_folder

## Checking the device

The SAM2 model needs a GPU to run otherwise it will be very slow and might even crash.

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla P100-PCIE-16GB


In [5]:
import os
os.path.exists("/kaggle/working/segment-anything-2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml")

True

## Initializing the SAM2 model and the model weights

*Be careful to choose a absolute path to the checkpoint and a relative path to the configs otherwise Hydra will not be able to find the checkpoint and the model will not be loaded.*

> Then, we load the model.



In [6]:
# Use the correct config path format
model_cfg = "configs/sam2.1/sam2.1_hiera_b+.yaml"
sam2_checkpoint = "/kaggle/working/segment-anything-2/checkpoints/sam2.1_hiera_base_plus.pt"

#Use this checkpoint and config instead of hiera_base_plus if you have enough GPU in order to get a better masks quality
#sam2_checkpoint = "/kaggle/working/segment-anything-2/checkpoints/sam2.1_hiera_large.pt"
#"model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

device = "cuda" if torch.cuda.is_available() else "cpu"
mask_generator = load_sam2_model(sam2_checkpoint, model_cfg, device=device)

## Creating the segmentation masks and saving the results

> The following cell creates the segmentation masks and saves the results in `output_masks/`. You can change the path to save the results in another folder.

> Change the path `image_folder` in order to fit with the right path to your dataset.

In [7]:
image_folder = "/kaggle/input/datasets/clementmartin1/flowers/Leroy_Merlin/Hypericum_perforatum"

mask_output = "../output_masks/masks/Hypericum_perforatum/"
overlay_output = "../output_masks/mask_overlays/Hypericum_perforatum/"

process_folder(
    image_folder,
    mask_generator,
    mask_output,
    overlay_output,
    only_best_mask=False
)

Processing Hypericum perforatum_26448568.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_26448568_overlay.png
Processing Hypericum perforatum_36155599.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_36155599_overlay.png
Processing Hypericum perforatum_38007374.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_38007374_overlay.png
Processing Hypericum perforatum_6421369.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_6421369_overlay.png
Processing Hypericum perforatum_29698850.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_29698850_overlay.png
Processing Hypericum perforatum_13038994.jpg
Saved overlay → ../output_masks/mask_overlays/Hypericum_perforatum/Hypericum perforatum_13038994_overlay.png
Processing Hypericum perforatum_63325919.jpg
Saved overlay → ../output_masks/m